# Reference run: reproduce every published benchmark
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/reference_run.ipynb)

Runs every pipeline at the reference setup, reconciles each model to its paper's table (`conf/base/parameters_benchmarks.yml`), refreshes the lesson figures and numbers and zips the results. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU): the frequency networks fit 15 seeds each. Results come back to the repo as a PR.

_Runs on a Colab **GPU runtime** (T4), selected automatically when you open it; if Colab gives you a CPU runtime, the first cell stops and says how to switch._

In [ ]:
# Setup: works in Colab (GPU runtime), the dev container and a local clone.
import os, sys
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

if "google.colab" in sys.modules and not os.path.exists("/content/deep-actuarial-science"):
    !git clone -q https://github.com/actuarialsociety/deep-actuarial-science /content/deep-actuarial-science
    !pip install -q -e /content/deep-actuarial-science
if "google.colab" in sys.modules:
    os.chdir("/content/deep-actuarial-science")
    sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

## 1. Run every pipeline
Frequency is the long one (15 seeds for each network).

In [ ]:
for p in ("frequency", "mortality", "reserving"):
    run(p)

## 2. Reconciliation: every model against its paper

In [ ]:
import glob, pandas as pd
rec = pd.concat(pd.read_csv(f) for f in sorted(glob.glob('data/08_reporting/*/*/reconciliation.csv')))
rec[['model', 'metric', 'expected', 'actual', 'tolerance', 'status', 'source']]

## 3. Refresh the lesson figures and numbers

In [ ]:
!python scripts/export_figures.py
!python scripts/fill_results.py

## 4. Download and open a PR
Unzip `reference_run.zip` at the repo root, check `git diff lessons/`, then commit on a branch and open a PR.

In [ ]:
import shutil, sys
shutil.make_archive('reference_run', 'zip', '.', 'lessons')
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('reference_run.zip')

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. In Colab the UI opens below and in a new tab through Colab's proxy (MLflow runs on the Colab machine, so a plain localhost link would point at your own computer).

In [ ]:
mlflow_ui()